In [38]:
import json
from pathlib import Path
import numpy as np


CAL_TABLE = Path(
    "/home/yousef/Antigravity_workspace/Calib_GE/Final/usrp_b200_calibration_table.json"
)

with CAL_TABLE.open("r", encoding="utf-8") as f:
    calibration_table = json.load(f)

print("Table chargée")

Table chargée


In [39]:
def dbfs_to_dbm(dbfs, freq_mhz, gain_db, table=calibration_table):
    """
    Convertit dBFS -> dBm à partir de la table JSON.

    Parameters
    ----------
    dbfs : float
        Valeur mesurée en dBFS.

    freq_mhz : float
        Fréquence en MHz.
        Exemple : 500.0

    gain_db : float
        Gain RX en dB.
        Exemple : 40.0 ou 76.0

    Returns
    -------
    float
        Puissance estimée en dBm.
    """

    frequencies = np.asarray(
        table["metadata"]["frequencies_mhz"],
        dtype=float
    )

    gains = np.asarray(
        table["metadata"]["gain_db_list"],
        dtype=float
    )

    grid = table["grid"]

    # ========================================================
    # Vérification des limites
    # ========================================================

    if freq_mhz < frequencies.min() or freq_mhz > frequencies.max():
        raise ValueError(
            f"Fréquence {freq_mhz} MHz hors calibration "
            f"[{frequencies.min()}, {frequencies.max()}] MHz"
        )

    if gain_db < gains.min() or gain_db > gains.max():
        raise ValueError(
            f"Gain {gain_db} dB hors calibration "
            f"[{gains.min()}, {gains.max()}] dB"
        )

    # ========================================================
    # Fréquences autour de la fréquence demandée
    # ========================================================

    i_high = np.searchsorted(
        frequencies,
        freq_mhz
    )

    if i_high == 0:
        f0 = f1 = frequencies[0]

    elif i_high == len(frequencies):
        f0 = f1 = frequencies[-1]

    elif np.isclose(
        frequencies[i_high],
        freq_mhz
    ):
        f0 = f1 = frequencies[i_high]

    else:
        f0 = frequencies[i_high - 1]
        f1 = frequencies[i_high]

    # ========================================================
    # Gains autour du gain demandé
    # ========================================================

    j_high = np.searchsorted(
        gains,
        gain_db
    )

    if j_high == 0:
        g0 = g1 = gains[0]

    elif j_high == len(gains):
        g0 = g1 = gains[-1]

    elif np.isclose(
        gains[j_high],
        gain_db
    ):
        g0 = g1 = gains[j_high]

    else:
        g0 = gains[j_high - 1]
        g1 = gains[j_high]

    # ========================================================
    # Coefficients d'interpolation
    # ========================================================

    if f1 == f0:
        tf = 0.0
    else:
        tf = (
            (freq_mhz - f0)
            /
            (f1 - f0)
        )

    if g1 == g0:
        tg = 0.0
    else:
        tg = (
            (gain_db - g0)
            /
            (g1 - g0)
        )

    # ========================================================
    # Lecture des 4 coins
    # ========================================================

    e00 = grid[f"{f0:.1f}"][f"{g0:.1f}"]
    e01 = grid[f"{f0:.1f}"][f"{g1:.1f}"]
    e10 = grid[f"{f1:.1f}"][f"{g0:.1f}"]
    e11 = grid[f"{f1:.1f}"][f"{g1:.1f}"]

    # ========================================================
    # Interpolation bilinéaire
    # ========================================================

    def interp(key):

        v00 = float(e00[key])
        v01 = float(e01[key])
        v10 = float(e10[key])
        v11 = float(e11[key])

        v_f0 = (
            (1 - tg) * v00
            +
            tg * v01
        )

        v_f1 = (
            (1 - tg) * v10
            +
            tg * v11
        )

        return (
            (1 - tf) * v_f0
            +
            tf * v_f1
        )

    slope_a = interp("slope_a")
    intercept_b = interp("intercept_b")

    # ========================================================
    # Conversion
    # ========================================================

    dbm = (
        slope_a * float(dbfs)
        +
        intercept_b
    )

    return float(dbm)

In [52]:
dbm = dbfs_to_dbm(
    dbfs=-17.28,
    freq_mhz=100.0,
    gain_db=40.0
)

print(f"Puissance = {dbm:.2f} dBm")

Puissance = -52.45 dBm


In [53]:
dbm = dbfs_to_dbm(
    dbfs=-9.0,
    freq_mhz=100.0,
    gain_db=60.0
)

print(f"Puissance = {dbm:.2f} dBm")

Puissance = -62.12 dBm
